In [ ]:
!kaggle datasets download -d rtlmhjbn/ip02-dataset

Dataset URL: https://www.kaggle.com/datasets/rtlmhjbn/ip02-dataset
License(s): copyright-authors
100% 2.94G/2.94G [01:23<00:00, 37.8MB/s]



In [ ]:
!unzip ip02-dataset.zip -d insect_dataset

Streaming output truncated to the last 5000 lines.
  inflating: insect_dataset/classification/val/29/23668.jpg  
  inflating: insect_dataset/classification/val/29/23674.jpg  
  inflating: insect_dataset/classification/val/29/23687.jpg  
  inflating: insect_dataset/classification/val/29/23689.jpg  
  inflating: insect_dataset/classification/val/29/23690.jpg  
  inflating: insect_dataset/classification/val/29/23699.jpg  
  inflating: insect_dataset/classification/val/29/23704.jpg  
  inflating: insect_dataset/classification/val/29/23707.jpg  
  inflating: insect_dataset/classification/val/29/23712.jpg  
  inflating: insect_dataset/classification/val/29/23716.jpg  
  inflating: insect_dataset/classification/val/29/23745.jpg  
  inflating: insect_dataset/classification/val/29/23746.jpg  
  inflating: insect_dataset/classification/val/29/23747.jpg  
  inflating: insect_dataset/classification/val/29/23771.jpg  
  inflating: insect_dataset/classification/val/29/23792.jpg  
  inflating: insect

In [ ]:
import numpy as np
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor

import plotly.graph_objs as go
import plotly.subplots as sp
from plotly.subplots import make_subplots
from plotly.offline import init_notebook_mode
init_notebook_mode(connected=True)

import os
import random
from PIL import Image

import warnings
warnings.filterwarnings("ignore")



In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense ,Conv2D,MaxPooling2D,Flatten,Dropout,BatchNormalization
import matplotlib.pyplot as plt
from tensorflow.keras.layers import GlobalAveragePooling2D
from keras.callbacks import EarlyStopping, ModelCheckpoint,ReduceLROnPlateau


In [ ]:
from pathlib import Path

data_dir = Path('/content/insect_dataset/classification/test')
class_name = [item.name for item in data_dir.glob('*') if item.is_dir()]
print(class_name)

['62', '17', '80', '78', '37', '100', '12', '11', '0', '66', '27', '101', '47', '79', '87', '25', '48', '52', '86', '13', '26', '21', '3', '7', '97', '92', '14', '33', '6', '89', '60', '16', '34', '64', '40', '82', '81', '59', '41', '75', '31', '76', '1', '56', '83', '23', '36', '39', '54', '58', '55', '28', '94', '85', '93', '20', '96', '5', '88', '68', '50', '98', '29', '77', '74', '67', '84', '22', '2', '69', '61', '8', '18', '32', '91', '49', '70', '45', '42', '30', '38', '65', '53', '10', '43', '63', '95', '35', '9', '73', '99', '90', '4', '15', '57', '51', '72', '24', '19', '44', '71', '46']


In [ ]:
train = tf.keras.utils.image_dataset_from_directory(
    '/content/insect_dataset/classification/train',
    labels = 'inferred',
    label_mode = 'categorical',
    batch_size = 128,
    image_size = (224, 224),
    class_names=class_name,
    shuffle = True,
    validation_split = 0,
    crop_to_aspect_ratio = True
)

Found 45095 files belonging to 102 classes.


In [ ]:
validation = tf.keras.utils.image_dataset_from_directory(
    '/content/insect_dataset/classification/val',
    labels = 'inferred',
    label_mode = 'categorical',
    batch_size = 128,
    image_size = (224, 224),
    class_names=class_name,
    shuffle = True,
    validation_split = 0,
    crop_to_aspect_ratio = True
)

Found 7508 files belonging to 102 classes.


In [ ]:
test = tf.keras.utils.image_dataset_from_directory(
    '/content/insect_dataset/classification/test',
    labels = 'inferred',
    label_mode = 'categorical',
    batch_size = 128,
    image_size = (224, 224),
    class_names=class_name,
    shuffle = True,
    validation_split = 0,
    crop_to_aspect_ratio = True
)

Found 22619 files belonging to 102 classes.


In [ ]:
# Performance: prefetch so the GPU isn't waiting on disk I/O
AUTOTUNE = tf.data.AUTOTUNE
train = train.prefetch(buffer_size=AUTOTUNE)
validation = validation.prefetch(buffer_size=AUTOTUNE)
test = test.prefetch(buffer_size=AUTOTUNE)


In [ ]:
for images, labels in train.take(1):
    print("Pixel value range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))
    print("Expected: roughly 0 to 255 (NOT 0 to 1)")
    print("Batch shape:", images.shape, "| Label shape:", labels.shape)


Pixel value range: 0.0 to 255.0
Expected: roughly 0 to 255 (NOT 0 to 1)
Batch shape: (128, 224, 224, 3) | Label shape: (128, 102)


In [ ]:

# Creating data augmentation pipeline
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomContrast(0.1)
])



In [ ]:
IMG_SIZE = (224, 224)
NUM_CLASSES = 102

In [ ]:

base_model = tf.keras.applications.EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=IMG_SIZE + (3,),
)
base_model.trainable = False  # freeze pretrained weights for fast initial training

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = data_augmentation(inputs)
x = tf.keras.applications.efficientnet.preprocess_input(x)  # ONLY normalization step
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)

model.summary()


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 102)            │       130,662 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,185,353 (15.97 MB)

 Trainable params: 133,222 (520.40 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(
            k=5,
            name='top5_acc'
        )
    ]
)

In [ ]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 102)            │       130,662 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,185,353 (15.97 MB)

 Trainable params: 133,222 (520.40 KB)

 Non-trainable params: 4,052,131 (15.46 MB)

In [ ]:
import tensorflow as tf

callbacks = [

    # Save only the best model
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model.keras",
        monitor="val_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    # Stop training when validation accuracy stops improving
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        mode="max",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),

    # Reduce learning rate when validation loss plateaus
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

In [17]:
history = model.fit(
    train,
    validation_data=validation,
    epochs=10,
    callbacks=callbacks
)

Epoch 1/10
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 274ms/step - accuracy: 0.2959 - loss: 3.3675 - top5_acc: 0.5086
Epoch 1: val_accuracy improved from None to 0.53436, saving model to best_model.keras

Epoch 1: finished saving model to best_model.keras
353/353 ━━━━━━━━━━━━━━━━━━━━ 136s 338ms/step - accuracy: 0.3897 - loss: 2.7116 - top5_acc: 0.6377 - val_accuracy: 0.5344 - val_loss: 1.8308 - val_top5_acc: 0.7940 - learning_rate: 0.0010
Epoch 2/10
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 305ms/step - accuracy: 0.4961 - loss: 2.0237 - top5_acc: 0.7641
Epoch 2: val_accuracy improved from 0.53436 to 0.57206, saving model to best_model.keras

Epoch 2: finished saving model to best_model.keras
353/353 ━━━━━━━━━━━━━━━━━━━━ 142s 357ms/step - accuracy: 0.5054 - loss: 1.9776 - top5_acc: 0.7722 - val_accuracy: 0.5721 - val_loss: 1.6776 - val_top5_acc: 0.8254 - learning_rate: 0.0010
Epoch 3/10
353/353 ━━━━━━━━━━━━━━━━━━━━ 0s 310ms/step - accuracy: 0.5271 - loss: 1.8474 - top5_acc: 0.7941
Epoch 3: val_accuracy impr

In [25]:
test_loss, test_acc, test_top5 = model.evaluate(test)

print("Test Accuracy:", test_acc)
print("Test Top-5 Accuracy:", test_top5)


177/177 ━━━━━━━━━━━━━━━━━━━━ 55s 309ms/step - accuracy: 0.6110 - loss: 1.5056 - top5_acc: 0.8500
Test Accuracy: 0.6110349893569946
Test Top-5 Accuracy: 0.8500375747680664


In [18]:
model.save('model.keras')



In [19]:
train = tf.keras.utils.image_dataset_from_directory('/content/insect_dataset/classification/train')

Found 45095 files belonging to 102 classes.


In [20]:
import json

class_names = train.class_names

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

In [21]:
with open("class_names.json") as f:
    class_names = json.load(f)

In [22]:
from google.colab import files

files.download("best_model.keras")
files.download("class_names.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download("class_mapping.json")

In [ ]:
model.save(model.h5)